### Hive

In [1]:
#pip install pandas

In [2]:
#pip install pyspark

In [3]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
#pip install spark

In [5]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/28 03:30:56 WARN Utils: Your hostname, fares-ali-HP-ZBook-15-G3, resolves to a loopback address: 127.0.1.1; using 172.18.0.1 instead (on interface br-1ebd6cf6ef84)
26/10/28 03:30:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/fares-ali/anaconda3/envs/data_engineering/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/28 03:30:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/28 03

In [6]:
employees = [{'id': 1, 'name': 'nada', 'age': 19, 'city': 'cairo'},
    {'id': 2, 'name': 'yara', 'age': 20, 'city': 'alexandria'},
    {'id': 3, 'name': 'youssef', 'age': 21, 'city': 'giza'}
    ]

In [7]:
schema = 'id string, name string, age string, city string'
df = spark.createDataFrame(employees, schema)
df.show()


+---+-------+---+----------+
| id|   name|age|      city|
+---+-------+---+----------+
|  1|   nada| 19|     cairo|
|  2|   yara| 20|alexandria|
|  3|youssef| 21|      giza|
+---+-------+---+----------+



In [8]:
spark.stop()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("HDFS_Write")
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .getOrCreate()
)

employees = [
    {"id": 1, "name": "nada", "age": 19, "city": "cairo"},
    {"id": 2, "name": "yara", "age": 20, "city": "alexandria"},
    {"id": 3, "name": "youssef", "age": 21, "city": "giza"},
]
schema = "id string, name string, age string, city string"
df = spark.createDataFrame(employees, schema)


df.write.mode("overwrite").parquet("hdfs://namenode:8020/session/employees")

26/10/28 03:31:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [9]:
df.write.mode("overwrite").parquet("hdfs://namenode:8020/session/employees")

In [10]:
spark.read.parquet('hdfs://namenode:8020/session/employees').show()

+---+-------+---+----------+
| id|   name|age|      city|
+---+-------+---+----------+
|  2|   yara| 20|alexandria|
|  3|youssef| 21|      giza|
|  1|   nada| 19|     cairo|
+---+-------+---+----------+



In [11]:
df.write.mode('overwrite').partitionBy('city').parquet('hdfs://namenode:8020/session/employees2')

In [12]:
spark.read.parquet('hdfs://namenode:8020/session/employees').show()

+---+-------+---+----------+
| id|   name|age|      city|
+---+-------+---+----------+
|  2|   yara| 20|alexandria|
|  3|youssef| 21|      giza|
|  1|   nada| 19|     cairo|
+---+-------+---+----------+



In [13]:
print("Spark Master:", spark.sparkContext.master)

print(
    "Hadoop defaultFS:",
    spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")
)

print(
    "NameNode DNS:",
    spark._jvm.java.net.InetAddress.getByName("namenode").getHostAddress()
)

Spark Master: local[*]
Hadoop defaultFS: file:///
NameNode DNS: 127.0.0.1


### Bronze Layer

In [14]:
pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pymongo
from pymongo import MongoClient

In [16]:
clinet = MongoClient('mongodb://localhost:27019/')

In [17]:
db = clinet['test']

In [18]:
data_students = list(db.students.find({}))

In [19]:
schema = 'student_id string, name string, age string, city string, gpa string, department string'

In [20]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StudentsData") \
    .getOrCreate()

26/10/28 03:31:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [21]:
import pyspark

data_students = list(db.students.find({}))
df_students = spark.createDataFrame(data_students, schema)

In [22]:
df_students .show()

+----------+-------+---+----------+---+----------------+
|student_id|   name|age|      city|gpa|      department|
+----------+-------+---+----------+---+----------------+
|         1|Mohamed| 22|     Cairo|3.5|Computer Science|
|         2|  Ahmed| 23|      Giza|3.2|     Engineering|
|         3|    Mai| 21|Alexandria|3.8|        Medicine|
|         4|   Sara| 24|     Cairo|3.6|Computer Science|
|         5|   Omar| 22|     Luxor|3.1|        Business|
|         6|Youssef| 20|     Aswan|3.4|     Engineering|
|         7|   Hana| 19|     Cairo|3.9|        Medicine|
|         8|  Tarek| 25|      Giza|  3|        Business|
|         9|   Nour| 22|Alexandria|3.7|Computer Science|
|        10|  Laila| 23|     Cairo|3.5|     Engineering|
|        11| Bassem| 27|     Luxor|3.3|        Business|
|        12|  Farah| 21|     Aswan|3.6|        Medicine|
|        13|  Karim| 24|      Giza|3.2|     Engineering|
|        14|  Salma| 20|     Cairo|3.8|Computer Science|
|        15|   Yara| 26|Alexand

In [23]:
data_grades = list(db.grades.find({}))
data_grades

[{'_id': ObjectId('6adfe4f4cbe08029da2dcb46'),
  'student_id': 1,
  'math': 90,
  'python': 87},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb47'),
  'student_id': 2,
  'math': 80,
  'python': 76},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb48'),
  'student_id': 3,
  'math': 84,
  'python': 88},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb49'),
  'student_id': 4,
  'math': 75,
  'python': 90},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb4a'),
  'student_id': 5,
  'math': 70,
  'python': 95},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb4b'),
  'student_id': 6,
  'math': 70,
  'python': 88},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb4c'),
  'student_id': 7,
  'math': 95,
  'python': 85},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb4d'),
  'student_id': 8,
  'math': 60,
  'python': 72},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb4e'),
  'student_id': 9,
  'math': 83,
  'python': 92},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcb4f'),
  'student_id': 10,
  'math': 78,
  'python': 81},
 {'_id': ObjectId('6adfe4f4cb

In [24]:
schema = 'student_id string, math string, python string'

In [25]:
df_grades = spark.createDataFrame(data_grades, schema)
df_students.show()

+----------+-------+---+----------+---+----------------+
|student_id|   name|age|      city|gpa|      department|
+----------+-------+---+----------+---+----------------+
|         1|Mohamed| 22|     Cairo|3.5|Computer Science|
|         2|  Ahmed| 23|      Giza|3.2|     Engineering|
|         3|    Mai| 21|Alexandria|3.8|        Medicine|
|         4|   Sara| 24|     Cairo|3.6|Computer Science|
|         5|   Omar| 22|     Luxor|3.1|        Business|
|         6|Youssef| 20|     Aswan|3.4|     Engineering|
|         7|   Hana| 19|     Cairo|3.9|        Medicine|
|         8|  Tarek| 25|      Giza|  3|        Business|
|         9|   Nour| 22|Alexandria|3.7|Computer Science|
|        10|  Laila| 23|     Cairo|3.5|     Engineering|
|        11| Bassem| 27|     Luxor|3.3|        Business|
|        12|  Farah| 21|     Aswan|3.6|        Medicine|
|        13|  Karim| 24|      Giza|3.2|     Engineering|
|        14|  Salma| 20|     Cairo|3.8|Computer Science|
|        15|   Yara| 26|Alexand

In [26]:
data_attendaces = list(db.attendances.find({}))
data_attendaces

[{'_id': ObjectId('6adfe4f4cbe08029da2dcade'),
  'student_id': 1,
  'date': '2026-10-01',
  'status': 'Present'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcadf'),
  'student_id': 2,
  'date': '2026-10-01',
  'status': 'Absent'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcae0'),
  'student_id': 3,
  'date': '2026-10-01',
  'status': 'Present'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcae1'),
  'student_id': 4,
  'date': '2026-10-01',
  'status': 'Present'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcae2'),
  'student_id': 5,
  'date': '2026-10-01',
  'status': 'Absent'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcae3'),
  'student_id': 6,
  'date': '2026-10-01',
  'status': 'Present'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcae4'),
  'student_id': 7,
  'date': '2026-10-01',
  'status': 'Present'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcae5'),
  'student_id': 8,
  'date': '2026-10-01',
  'status': 'Absent'},
 {'_id': ObjectId('6adfe4f4cbe08029da2dcae6'),
  'student_id': 9,
  'date': '2026-10-01',
 

In [27]:
schema = 'student_id string, date string, status string'

In [28]:
df_attendances = spark.createDataFrame(data_attendaces, schema)
df_attendances.show()

+----------+----------+-------+
|student_id|      date| status|
+----------+----------+-------+
|         1|2026-10-01|Present|
|         2|2026-10-01| Absent|
|         3|2026-10-01|Present|
|         4|2026-10-01|Present|
|         5|2026-10-01| Absent|
|         6|2026-10-01|Present|
|         7|2026-10-01|Present|
|         8|2026-10-01| Absent|
|         9|2026-10-01|Present|
|        10|2026-10-01|Present|
|        11|2026-10-02|Present|
|        12|2026-10-02| Absent|
|        13|2026-10-02|Present|
|        14|2026-10-02|Present|
|        15|2026-10-02| Absent|
|        16|2026-10-02|Present|
|        17|2026-10-02|Present|
|        18|2026-10-02| Absent|
|        19|2026-10-02|Present|
|        20|2026-10-02|Present|
+----------+----------+-------+
only showing top 20 rows


In [29]:
df_students
df_grades
df_attendances

DataFrame[student_id: string, date: string, status: string]

In [30]:
df_students.write.mode('overwrite').parquet('hdfs://namenode:8020/project/bronze/students')
df_grades.write.mode('overwrite').parquet('hdfs://namenode:8020/project/bronze/grades')
df_attendances.write.mode('overwrite').parquet('hdfs://namenode:8020/project/bronze/attendances')

26/10/28 03:31:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/28 03:31:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


### Silver Layer

In [31]:
df_students = spark.read.parquet('hdfs://namenode:8020/project/bronze/students')
df_grades = spark.read.parquet('hdfs://namenode:8020/project/bronze/grades')
df_attendances = spark.read.parquet('hdfs://namenode:8020/project/bronze/attendances')

In [32]:
df_students.show(5)

+----------+-----------+---+----------+---+----------------+
|student_id|       name|age|      city|gpa|      department|
+----------+-----------+---+----------+---+----------------+
|        60|     Yasmin| 25|     Luxor|3.9|     Engineering|
|        61|       Zain| 19|     Aswan|3.3|        Medicine|
|        62|Abdelrahman| 22|     Cairo|3.3|        Business|
|        63|      Basma| 23|      Giza|3.7|Computer Science|
|        64|     Cyrine| 21|Alexandria|3.8|     Engineering|
+----------+-----------+---+----------+---+----------------+
only showing top 5 rows


In [33]:
df_students.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- city: string (nullable = true)
 |-- gpa: string (nullable = true)
 |-- department: string (nullable = true)



In [34]:
df_students = df_students.withColumn('age', df_students['age'].cast('int'))
df_students = df_students.withColumn('student_id', df_students['student_id'].cast('int'))
df_students = df_students.withColumn('gpa', df_students['gpa'].cast('float'))


In [35]:
df_students.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- gpa: float (nullable = true)
 |-- department: string (nullable = true)



In [36]:
df_students = df_students.filter(df_students['age'] > 0)

In [37]:
df_students.show(5)

+----------+-----------+---+----------+---+----------------+
|student_id|       name|age|      city|gpa|      department|
+----------+-----------+---+----------+---+----------------+
|        60|     Yasmin| 25|     Luxor|3.9|     Engineering|
|        61|       Zain| 19|     Aswan|3.3|        Medicine|
|        62|Abdelrahman| 22|     Cairo|3.3|        Business|
|        63|      Basma| 23|      Giza|3.7|Computer Science|
|        64|     Cyrine| 21|Alexandria|3.8|     Engineering|
+----------+-----------+---+----------+---+----------------+
only showing top 5 rows


In [38]:
df_grades.show(5)

+----------+----+------+
|student_id|math|python|
+----------+----+------+
|        64|  73|    77|
|        65|  75|    80|
|        66|  67|    71|
|        67|  86|    93|
|        68|  75|    80|
+----------+----+------+
only showing top 5 rows


In [39]:
df_grades.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- math: string (nullable = true)
 |-- python: string (nullable = true)



In [40]:
df_grades =  df_grades.withColumn('student_id', df_grades['student_id'].cast('int'))
df_grades =  df_grades.withColumn('math', df_grades['math'].cast('int'))
df_grades =  df_grades.withColumn('python', df_grades['python'].cast('int'))


In [41]:
df_grades.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- math: integer (nullable = true)
 |-- python: integer (nullable = true)



In [42]:
df_attendances.show(5)

+----------+----------+-------+
|student_id|      date| status|
+----------+----------+-------+
|        66|2026-10-08| Absent|
|        67|2026-10-08|Present|
|        68|2026-10-08|Present|
|        69|2026-10-08| Absent|
|        70|2026-10-09|Present|
+----------+----------+-------+
only showing top 5 rows


In [43]:
df_attendances.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- status: string (nullable = true)



In [44]:
df_attendances = df_attendances.withColumn('student_id', df_attendances['student_id'].cast('int'))
df_attendances = df_attendances.withColumn('date', df_attendances['date'].cast('date'))

df_attendances.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- status: string (nullable = true)



In [45]:
df_students.write.mode('overwrite').parquet('hdfs://namenode:8020/project/silver/students')
df_grades.write.mode('overwrite').parquet('hdfs://namenode:8020/project/silver/grades')
df_attendances.write.mode('overwrite').parquet('hdfs://namenode:8020/project/silver/attendances')

26/10/28 03:31:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/28 03:31:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/28 03:31:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


### Gold Layer

In [46]:
df_students = spark.read.parquet('hdfs://namenode:8020/project/silver/students')
df_grades = spark.read.parquet('hdfs://namenode:8020/project/silver/grades')
df_attendances = spark.read.parquet('hdfs://namenode:8020/project/silver/attendances')

In [47]:
df_fact = df_grades.join(df_attendances, on = 'student_id', how='inner')
df_fact = df_fact.join(df_students, on = 'student_id', how='inner')

df_fact.show(5)

+----------+----+------+----------+------+------+---+----------+---+-----------+
|student_id|math|python|      date|status|  name|age|      city|gpa| department|
+----------+----+------+----------+------+------+---+----------+---+-----------+
|        64|  73|    77|2026-10-08|Absent|Cyrine| 21|Alexandria|3.8|Engineering|
|        64|  73|    77|2026-10-08|Absent|Cyrine| 21|Alexandria|3.8|Engineering|
|        64|  73|    77|2026-10-08|Absent|Cyrine| 21|Alexandria|3.8|Engineering|
|        64|  73|    77|2026-10-08|Absent|Cyrine| 21|Alexandria|3.8|Engineering|
|        64|  73|    77|2026-10-08|Absent|Cyrine| 21|Alexandria|3.8|Engineering|
+----------+----+------+----------+------+------+---+----------+---+-----------+
only showing top 5 rows


In [48]:
df_fact = df_fact.select ('student_id', 'name', 'department', 'age', 'city', 'date', 'status', 'math', 'python', 'gpa')

In [49]:
df_fact.show(5)

+----------+------+-----------+---+----------+----------+------+----+------+---+
|student_id|  name| department|age|      city|      date|status|math|python|gpa|
+----------+------+-----------+---+----------+----------+------+----+------+---+
|        64|Cyrine|Engineering| 21|Alexandria|2026-10-08|Absent|  73|    77|3.8|
|        64|Cyrine|Engineering| 21|Alexandria|2026-10-08|Absent|  73|    77|3.8|
|        64|Cyrine|Engineering| 21|Alexandria|2026-10-08|Absent|  73|    77|3.8|
|        64|Cyrine|Engineering| 21|Alexandria|2026-10-08|Absent|  73|    77|3.8|
|        64|Cyrine|Engineering| 21|Alexandria|2026-10-08|Absent|  73|    77|3.8|
+----------+------+-----------+---+----------+----------+------+----+------+---+
only showing top 5 rows


In [50]:
df_fact.write.mode('overwrite').parquet('hdfs://namenode:8020/project/gold/fact')

26/10/28 03:32:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [51]:

df_fact.toPandas().to_csv('fact.csv', index=False)

/home/fares-ali/anaconda3/envs/data_engineering/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
